# Data Cleaning

### Libraries

In [3]:
import pandas as pd
import numpy as np

### Team Statistics

In [ ]:
team_stats = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/team_stats.csv')

team_stats = team_stats.drop(columns='Unnamed: 0')
team_stats = team_stats.rename(columns={'Cmp-Att-Yd-TD-INT':'ManyStats'})
team_stats[['Completion', 'Attempts', 'Yards', 'Touchdowns', 'Interceptions']] = team_stats.ManyStats.str.split('-', expand=True)
team_stats = team_stats.drop(columns='ManyStats')

team_stats.columns = team_stats.columns.str.replace(' ', '_')
team_stats.columns = team_stats.columns.str.replace('.', '_')
team_stats.columns = team_stats.columns.str.replace('-', '_')

team_stats[['Success4thDown', 'Failed4thDown']] = team_stats.Fourth_Down_Conv_.str.split('-', expand=True)
team_stats[['Fumbles', 'FumblesLost']] = team_stats.Fumbles_Lost.str.split('-', expand=True)
team_stats[['Penalties', 'PenaltyYards']] = team_stats.Penalties_Yards.str.split('-', expand=True)
#team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats[['Sacked', 'SackYards']] = team_stats.Sacked_Yards.str.split('-', expand=True)
team_stats[['Success3rdDown', 'Failed3rdDown']] = team_stats.Third_Down_Conv_.str.split('-', expand=True)
team_stats[['TimeMinutes', 'TimeSeconds']] = team_stats.Time_of_Possession.str.split(':', expand=True)

team_stats = team_stats.drop(columns={'Fourth_Down_Conv_', 'Fumbles_Lost', 'Penalties_Yards', 'Sacked_Yards', 'Third_Down_Conv_', 'Time_of_Possession'})

team_stats['Rush_Yds_TDs'].str.count('-').value_counts()

team_stats.loc[2083]

team_stats.at[2083,'Rush_Yds_TDs'] = '8-neg1-0'

team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats = team_stats.drop(columns='Rush_Yds_TDs')

team_stats.at[2083, "RushYards"] = -1

skip_cols = ['team_name', 'date']
cols_to_change = [col for col in team_stats.columns if col not in skip_cols]
team_stats[cols_to_change] = team_stats[cols_to_change].astype(int)

team_stats['CompletionPct'] = round(100 * (team_stats['Completion'] / team_stats['Attempts']), 1)
team_stats['PctYardsPass'] = round(100 * (team_stats['Yards'] / team_stats['Total_Yards']), 1)
team_stats['PctYardsRush'] = round(100 * (team_stats['RushYards'] / team_stats['Total_Yards']), 1)
team_stats['SuccessRate4thDown'] = round(
    100 * (team_stats['Success4thDown'] / (team_stats['Success4thDown'] + team_stats['Failed4thDown'])), 1)
team_stats['SuccessRate3rdDown'] = round(
    100 * (team_stats['Success3rdDown'] / (team_stats['Success3rdDown'] + team_stats['Failed3rdDown'])), 1)
team_stats['PossessionTime'] = (60 * team_stats['TimeMinutes']) + team_stats['TimeSeconds']

team_stats[['DayofWeek', 'Month', 'DayofMonth', 'Year']] = team_stats.date.str.split(' ', expand=True)
team_stats['DayofMonth'] = team_stats['DayofMonth'].str.rstrip(',')

game_months = [
    (team_stats['Month'] == 'Sep'),
    (team_stats['Month'] == 'Oct'),
    (team_stats['Month'] == 'Nov'),
    (team_stats['Month'] == 'Dec'),
    (team_stats['Month'] == 'Jan'),
    (team_stats['Month'] == 'Feb')
]
month_choices = [9, 10, 11, 12, 1, 2]

team_stats['Month'] = np.select(game_months, month_choices, default=False)
team_stats['DayofMonth'] = team_stats['DayofMonth'].astype(int)
team_stats['Year'] = team_stats['Year'].astype(int)

date_mapping = {
    'Year':'year',
    'DayofMonth':'day',
    'Month':'month'}

team_stats = team_stats.rename(mapper=date_mapping, axis=1)
team_stats['gamedate'] = pd.to_datetime(team_stats[['year', 'month', 'day']])

team_stats = team_stats.drop(columns={'date','Completion','Success4thDown','Failed4thDown','Success3rdDown','Failed3rdDown'})

In [17]:
team_stats

,First_Downs,Net_Pass_Yards,Total_Yards,Turnovers,team_name,Attempts,Yards,Touchdowns,Interceptions,Fumbles,...,PctYardsPass,PctYardsRush,SuccessRate4thDown,SuccessRate3rdDown,PossessionTime,DayofWeek,month,day,year,gamedate
0,19,283,389,3,GNB,43,291,1,3,2,...,74.8,27.2,20.0,34.8,2082,Sunday,11,6,2022,06-11-2022
1,19,137,254,1,DET,26,137,2,1,0,...,53.9,46.1,0.0,35.3,1518,Sunday,11,6,2022,06-11-2022
2,20,127,338,2,PHI,28,154,1,1,2,...,45.6,62.4,50.0,15.8,2178,Sunday,12,22,2024,22-12-2024
3,23,255,368,5,WAS,39,258,5,2,3,...,70.1,30.7,40.0,35.0,1422,Sunday,12,22,2024,22-12-2024
4,23,204,311,0,DAL,41,204,2,0,0,...,65.6,34.4,NaN,30.4,1881,Sunday,11,19,2023,19-11-2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,16,119,193,1,PIT,31,148,1,0,2,...,76.7,38.3,0.0,25.0,1320,Saturday,1,4,2025,04-01-2025
2274,19,242,339,1,DAL,32,253,2,0,1,...,74.6,28.6,50.0,25.0,1862,Sunday,12,24,2023,24-12-2023
2275,22,284,375,0,MIA,37,293,1,0,1,...,78.1,24.3,0.0,31.6,1738,Sunday,12,24,2023,24-12-2023
2276,18,171,264,1,WAS,32,191,1,1,1,...,72.3,35.2,33.3,20.0,1616,Thursday,11,14,2024,14-11-2024


### Offensive Player Statistics

In [ ]:
player_offense = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/player_offense.csv')

new_offense_cols = ['randnumber','playername','team','completions','passattempts',
                    'passingyards','passingtds','interceptions','sackstaken','yardslostbysack',
                    'longestcompletion','passrating','rushingattempts','rushyards','rushtds',
                    'longestrush','targets','receptions','receivingyards','receivingtds',
                    'longestreception','fumbles','fumbleslost','gamedate']

player_offense.columns = new_offense_cols

realplayer = player_offense['playername'] != "Player"
playerhere = player_offense['playername'].notna()
player_offense = player_offense[realplayer]
player_offense = player_offense[playerhere]

player_offense = player_offense.drop(columns='randnumber')

player_offense[['dayofweek','month','day','year']] = player_offense.gamedate.str.split(' ', expand=True)
player_offense = player_offense.drop(columns='gamedate')
player_offense['day'] = player_offense['day'].str.rstrip(',')

game_months = [
    (player_offense['month'] == 'Sep'),
    (player_offense['month'] == 'Oct'),
    (player_offense['month'] == 'Nov'),
    (player_offense['month'] == 'Dec'),
    (player_offense['month'] == 'Jan'),
    (player_offense['month'] == 'Feb')
]

player_offense['month'] = np.select(game_months, month_choices, default=False)
player_offense['gamedate'] = pd.to_datetime(player_offense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_offense.columns if col not in skip_cols]
player_offense[cols_to_change] = player_offense[cols_to_change].astype(float)

player_offense['completionpct'] = round(100 * (player_offense['completions'] / player_offense['passattempts']), 1)
player_offense['td2int'] = round(player_offense['passingtds'] / player_offense['interceptions'], 3)
player_offense['yardsperrush'] = round(player_offense['rushyards'] / player_offense['rushingattempts'], 1)

/var/folders/xv/_79n8tfd6mx4j2z8m6rstygr0000gn/T/ipykernel_25368/4028261515.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  player_offense = player_offense[playerhere]


In [16]:
player_offense

,playername,team,completions,passattempts,passingyards,passingtds,interceptions,sackstaken,yardslostbysack,longestcompletion,...,fumbles,fumbleslost,dayofweek,month,day,year,gamedate,completionpct,td2int,yardsperrush
1,Aaron Rodgers,GNB,23.0,43.0,291.0,1.0,3.0,1.0,8.0,47.0,...,0.0,0.0,Sunday,11.0,6.0,2022.0,2022-11-06,53.5,0.333,10.0
2,AJ Dillon,GNB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,Sunday,11.0,6.0,2022.0,2022-11-06,NaN,NaN,3.1
3,Aaron Jones,GNB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,Sunday,11.0,6.0,2022.0,2022-11-06,NaN,NaN,2.8
4,Kylin Hill,GNB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,Sunday,11.0,6.0,2022.0,2022-11-06,NaN,NaN,7.0
5,Allen Lazard,GNB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,Sunday,11.0,6.0,2022.0,2022-11-06,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24799,Kenneth Gainwell,PHI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,Thursday,11.0,14.0,2024.0,2024-11-14,NaN,NaN,10.8
24800,A.J. Brown,PHI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,Thursday,11.0,14.0,2024.0,2024-11-14,NaN,NaN,NaN
24801,Dallas Goedert,PHI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,Thursday,11.0,14.0,2024.0,2024-11-14,NaN,NaN,NaN
24802,DeVonta Smith,PHI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,Thursday,11.0,14.0,2024.0,2024-11-14,NaN,NaN,NaN


### Defensive Player Statistics

### Advanced Passing Statistics

### Advanced Receiving Statistics

### Advanced Defensive Statistics